In [11]:
import minsearch
import pickle
import json

# 1. Prepare evaluation test dataset (Ground Truth)
eval_dataset = [
    {
        "query": "best exercises for bigger legs",
        "expected_doc_id": "doc_102",  # Doc ID known to hold the correct answer
        "ground_truth_answer": "Squats and leg presses are ideal for leg hypertrophy."
    },
    {
        "query": "I am sore after training",
        "expected_doc_id": "doc_205",
        "ground_truth_answer": "Ensure rest, hydration, light stretching, and adequate protein."
    }
]

# Load trained index
with open("index/fitness_index.pkl", "rb") as f:
    idx = pickle.load(f)

# --- EVALUATING RETRIEVAL ---
def evaluate_retrieval(test_set, top_k=2):
    hits = 0
    reciprocal_ranks = []

    for item in test_set:
        query = item["query"]
        expected_id = item["expected_doc_id"]

        # Run search
        results = idx.search(query=query, num_results=top_k)
        retrieved_ids = [doc["id"] for doc in results]

        # Check Hit Rate
        if expected_id in retrieved_ids:
            hits += 1
            rank = retrieved_ids.index(expected_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    hit_rate = hits / len(test_set)
    mrr = sum(reciprocal_ranks) / len(test_set)
    return hit_rate, mrr

hit_rate, mrr = evaluate_retrieval(eval_dataset, top_k=2)
print(f"Retrieval Hit Rate@2: {hit_rate:.2%}")
print(f"Retrieval MRR: {mrr:.2f}")


# --- EVALUATING END-TO-END RAG (LLM-as-a-Judge Pattern) ---
def evaluate_faithfulness(retrieved_context: str, llm_response: str) -> float:
    """
    Passes context and answer to an evaluator model to check for hallucinations.
    """
    eval_prompt = f"""
    Compare the following Answer with the Context provided.
    Does the Answer contain any statements or claims that are NOT supported by the Context?
    Respond with a score from 0.0 (completely hallucinated) to 1.0 (fully grounded/faithful).
    
    Context: {retrieved_context}
    Answer: {llm_response}
    
    Score (0.0 to 1.0):
    """
    # Send eval_prompt to an LLM evaluator (e.g. GPT-4, DeepEval, or RAGAS framework)
    # return float(llm_eval_score)
    return 1.0  # Placeholder score

Retrieval Hit Rate@2: 0.00%
Retrieval MRR: 0.00
